In [1]:
import openai

import json
import numpy as np
import os

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

from huggingface_hub import InferenceClient

/home/spellen/miniconda3/envs/apm_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the dataset
with open('ocpq_dataset.json', 'r') as f:
    dataset = json.load(f)

# Load the generated embeddings
embeddings = np.load('embeddings.npy')

# Load the local embedding model so we can embed user queries on the fly
model = SentenceTransformer('./local-all-MiniLM-L6-v2') 

def get_embedding(text):
    return model.encode(text)

# Initialize Hugging Face Inference Client for generation
hf_client = InferenceClient(api_key=os.getenv("HF_TOKEN"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1759.38it/s]
BertModel LOAD REPORT from: ./local-all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
def retrieve_examples(query, dataset, embeddings, top_k=3):
    """
    Retrieve the top-k most similar examples from the dataset based on cosine similarity of embeddings.
    """
    query_emb = get_embedding(query)
    
    # Calculate cosine similarity between the query and all dataset questions
    similarities = cosine_similarity([query_emb], embeddings)[0]
    
    # Get the indices of the top-k most similar questions
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    examples = []
    for idx in top_indices:
        examples.append({
            "nl": dataset[idx]["nl"],
            "query": dataset[idx]["query"]
        })
    
    return examples

In [ ]:
retrieve_examples("create order happens before ship order", dataset, embeddings)

In [16]:
def nl_to_ocpq_rag(query, dataset, embeddings):
    # Retrieve relevant examples
    examples = retrieve_examples(query, dataset, embeddings)
    
    # Construct a prompt with few-shot examples
    example_prompts = []
    for ex in examples:
        example_prompts.append(
            f"Question: {ex['nl']}\nJSON: {json.dumps(ex['query'])}"
        )
    few_shot_string = "\n\n".join(example_prompts)
    
    prompt = f"""
Translate natural language process queries into OCPQ JSON based on the examples. 
Output ONLY valid JSON.

EXAMPLES:
{few_shot_string}

Now translate this query:
Question: {query}
JSON:
"""

    # Using a free open-source model via Hugging Face Serverless API
    response = hf_client.chat_completion(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=[{"role":"user","content":prompt}],
        max_tokens=800,
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content
    
    try:
        # Llama-3 might add markdown quotes like ```json, so we strip them
        content = content.replace("```json", "").replace("```", "").strip()
        return json.loads(content)
    except json.JSONDecodeError:
        print("Failed to decode JSON. Raw output:")
        print(content)
        return None

# Test the function with a new query
# test_query = "merge pull request after close issue"
# print(json.dumps(nl_to_ocpq_rag(test_query, dataset, embeddings), indent=2))

In [23]:
nl_to_ocpq_rag("how do I check the cases where sign contract happens before fill in details", dataset, embeddings)

{'nodes': [{'Box': [{'newEventVars': {'0': ['sign contract'],
      '1': ['fill in details']},
     'newObjectVars': {'0': ['cases']},
     'filters': [],
     'sizeFilters': [],
     'constraints': [{'type': 'Filter',
       'filter': {'type': 'TimeBetweenEvents',
        'from_event': 0,
        'to_event': 1,
        'min_seconds': 0,
        'max_seconds': None}}],
     'evVarLabels': {},
     'obVarLabels': {}},
    []]}],
 'edgeNames': []}

In [24]:
nl_to_ocpq_rag("write me a poem about shipping order", dataset, embeddings)

{'nodes': [{'Box': [{'newEventVars': {'0': ['ship package']},
     'newObjectVars': {'0': ['poem']},
     'filters': [],
     'sizeFilters': [],
     'constraints': [],
     'evVarLabels': {},
     'obVarLabels': {}}]},
  []],
 'edgeNames': []}

In [14]:
def nl_to_ocpq(query):

    prompt = f"""
Translate natural language process queries into OCPQ JSON.
Output ONLY valid JSON.

Question: {query}
JSON:
"""

    response = hf_client.chat_completion(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=[{"role":"user","content":prompt}],
        max_tokens=800
    )

    content = response.choices[0].message.content
    
    try:
        content = content.replace("```json", "").replace("```", "").strip()
        return json.loads(content)
    except json.JSONDecodeError:
        print("Failed to decode JSON. Raw output:")
        print(content)
        return None